# Import Library

In [ ]:
from ultralytics import YOLO
import os
from glob import glob
import shutil
import cv2
# from kan import *
# from kan.utils import create_dataset
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import matplotlib.pyplot as plt


# Prepare Dataset

## Turn Video Image

In [ ]:
def turn_video_image(video_path: str, save_path: str):

    # Validate video file existence
    if not os.path.exists(video_path):
        print(f"Error: Video file not found at {video_path}")
        return

    # Open the video file
    video_capture = cv2.VideoCapture(video_path)

    if not video_capture.isOpened():
        print(f"Error: Could not open video file {video_path}. "
              "This might be due to an unsupported codec, corrupted file, or missing FFmpeg libraries.")
        return

    # Log video information
    original_fps = video_capture.get(cv2.CAP_PROP_FPS)
    total_frames_in_video = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Processing video: {os.path.basename(video_path)}")
    print(f"  Original video FPS: {original_fps}")
    print(f"  Total frames in video (reported): {total_frames_in_video}")
    print(f"  Desired extraction rate: 1 frame per second")

    # Define extraction interval (1 second = 1000 milliseconds)
    desired_interval_ms = 1000.0 # We want 1 frame every 1000 ms (1 second)

    # Initialize counters and flags
    next_frame_save_time_ms = 0.0 # Target timestamp for the next frame to save
    frames_saved = 0 # Count of successfully saved images

    # Prepare save directory
    if os.path.exists(save_path):
        shutil.rmtree(save_path) # Clear existing directory
    os.makedirs(save_path, exist_ok=True)
    print(f"  Saving extracted images to: {save_path}")

    # Main video processing loop
    while True:
        ret, frame = video_capture.read()

        # Check if frame was read successfully
        if not ret:
            print("  End of video stream or failed to read frame.")
            break

        # Get the current timestamp of the frame being processed
        current_timestamp_ms = video_capture.get(cv2.CAP_PROP_POS_MSEC)

        # Check if current frame's timestamp has reached or passed the target save time
        # Use a small epsilon (0.001) for float comparison to handle precision issues
        if current_timestamp_ms >= next_frame_save_time_ms - 0.001:
            # Calculate time components for filename (MM_SS_mmm.png)
            total_seconds_float = current_timestamp_ms / 1000.0
            minutes = int(total_seconds_float // 60)
            remaining_seconds_float = total_seconds_float % 60
            seconds = int(remaining_seconds_float)
            milliseconds = int((remaining_seconds_float - seconds) * 1000)

            image_filename = f"{minutes:02d}_{seconds:02d}_{milliseconds:03d}.png"
            output_path = os.path.join(save_path, image_filename)

            # Try to save the frame
            try:
                cv2.imwrite(output_path, frame)
                frames_saved += 1
                # Update the target time for the next frame
                next_frame_save_time_ms += desired_interval_ms
            except Exception as e:
                print(f"  Error saving frame at {current_timestamp_ms:.2f}ms to {output_path}: {e}")
                # Don't increment next_frame_save_time_ms if save failed, to retry with a nearby frame if possible
                pass

    # 8. Release video resources
    video_capture.release()
    print(f"  Successfully extracted {frames_saved} frames from {os.path.basename(video_path)}.")
    print("  Video processing complete for this file.")


In [ ]:
def turn_video_image_framerate(video_path: str, save_path: str, FRAME_RATE: int = 5):

    # ตรวจสอบว่าไฟล์วิดีโอมีอยู่จริงหรือไม่
    if not os.path.exists(video_path):
        print(f"Error: Video file not found at {video_path}")
        return

    video_capture = cv2.VideoCapture(video_path)

    if not video_capture.isOpened():
        print(f"Error: Could not open video file {video_path}. This might be due to unsupported codec or corrupted file.")
        print(f"Attempted to open: {video_path}")
        return

    original_fps = video_capture.get(cv2.CAP_PROP_FPS)
    total_frames_in_video = video_capture.get(cv2.CAP_PROP_FRAME_COUNT)
    video_duration_ms = video_capture.get(cv2.CAP_PROP_POS_MSEC) # Should be 0 at start

    print(f"Original video FPS: {original_fps}")
    print(f"Total frames in video (reported by OpenCV): {total_frames_in_video}")
    # print(f"Video duration (reported by OpenCV, might be inaccurate): {video_duration_ms / 1000:.2f} seconds") # This gets total duration only after reading all frames or seeking
    print(f"Desired FRAME_RATE for extraction: {FRAME_RATE}")

    desired_interval_ms = 1000.0 / FRAME_RATE if FRAME_RATE > 0 else 0

    if desired_interval_ms == 0:
        print("Error: FRAME_RATE must be greater than 0.")
        video_capture.release()
        return

    next_frame_save_time_ms = 0.0
    frames_saved = 0

    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    os.makedirs(save_path, exist_ok=True)
    print(f"Saving extracted images to: {save_path}")

    frame_read_counter = 0 # เพื่อติดตามว่าอ่านไปกี่เฟรมแล้ว

    while True:
        ret, frame = video_capture.read()
        frame_read_counter += 1

        if not ret:
            # หากอ่านไม่ได้เลยตั้งแต่แรก (frame_read_counter=1, ret=False) อาจเป็นปัญหาใหญ่
            if frame_read_counter == 1:
                print(f"Warning: Failed to read the first frame of {video_path}. Video might be empty or corrupted.")
            print("End of video stream or failed to read frame.")
            break

        current_timestamp_ms = video_capture.get(cv2.CAP_PROP_POS_MSEC)

        # Log ความคืบหน้า (เช่น ทุกๆ 100 เฟรมที่อ่านได้)
        if frame_read_counter % 100 == 0:
            print(f"  Processing frame {frame_read_counter} at {current_timestamp_ms:.2f}ms...")

        if current_timestamp_ms >= next_frame_save_time_ms - 0.001:
            total_seconds_float = current_timestamp_ms / 1000.0
            minutes = int(total_seconds_float // 60)
            remaining_seconds_float = total_seconds_float % 60
            seconds = int(remaining_seconds_float)
            milliseconds = int((remaining_seconds_float - seconds) * 1000)

            image_filename = f"{minutes:02d}_{seconds:02d}_{milliseconds:03d}.png"
            output_path = os.path.join(save_path, image_filename)

            try:
                cv2.imwrite(output_path, frame)
                frames_saved += 1
                next_frame_save_time_ms += desired_interval_ms
            except Exception as e:
                print(f"Error saving frame at {current_timestamp_ms:.2f}ms to {output_path}: {e}")
                pass

    video_capture.release()
    print(f"Successfully extracted {frames_saved} frames from {video_path}.")
    print("Video processing complete.")

In [ ]:
DATASET_PATH = "/project/ai901504-ai0004/kaggle_competition/week7/train"
SAVE_IMAGE_BASE_PATH = "/project/ai901504-ai0004/501641_Big/week7/image"
FRAME_RATE = 10

videos_to_process = [
    # คุณคมสันต์ ลี
    {
        'video_sub_path': 'คุณคมสันต์ ลี/Flash Express  The Secret Sauce Highlight.mp4',
        'save_folder_name': 'Flash Express The Secret Sauce Highlight'
    },
    # Rose Interview
    {
        'video_sub_path': 'Rose Interview/rose On Working With Bruno Mars, The Drinking Game That Inspired Her Single and Her New Album ‘Rosie’.mp4',
        'save_folder_name': 'rose On Working With Bruno Mars, The Drinking Game That Inspired Her Single and Her New Album ‘Rosie’'
    },
    # Gary Oldman Series
    {
        'video_sub_path': 'Gary Oldman/Gary Oldman Interview 2 (Descending).mp4',
        'save_folder_name': 'Gary Oldman Interview 2 (Descending)'
    },
    {
        'video_sub_path': 'Gary Oldman/Gary Oldman Interview 3 (Rising).mp4',
        'save_folder_name': 'Gary Oldman Interview 3 (Rising)'
    },
    {
        'video_sub_path': 'Gary Oldman/Gary Oldman Interview 4 (Increasing Pressure vs Decreasing Pressure).mp4',
        'save_folder_name': 'Gary Oldman Interview 4 (Increasing Pressure vs Decreasing Pressure)'
    },
    # Christianne Amanpour Interview
    {
        'video_sub_path': 'Christianne Amanpour Interview/Christianne Amanpour Interview.mp4',
        'save_folder_name': 'Christianne Amanpour Interview'
    },
    # Benedict Cumberbatch Interview (Path นี้เคยไม่มี DATASET_PATH นำหน้า)
    {
        'video_sub_path': 'Benedict Cumberbatch Interview/Benedict Cumberbatch Interview.mp4',
        'save_folder_name': 'Benedict Cumberbatch Interview'
    }
]


for video_info in tqdm(videos_to_process):
    video_full_path = os.path.join(DATASET_PATH, video_info['video_sub_path'])
    save_full_path = os.path.join(SAVE_IMAGE_BASE_PATH, video_info['save_folder_name'])
    
    turn_video_image_framerate(
        video_path=video_full_path,
        save_path=save_full_path,
        FRAME_RATE=FRAME_RATE
    )
    
    
for i in tqdm(range(1, 6)):
    video_name = f's{i:02d}_smiley.mp4'
    video_full_path = os.path.join(DATASET_PATH, video_name)
    save_folder_name = f's{i:02d}_smiley'
    save_full_path = os.path.join(SAVE_IMAGE_BASE_PATH, save_folder_name)
    
    turn_video_image_framerate(
        video_path=video_full_path,
        save_path=save_full_path,
        FRAME_RATE=FRAME_RATE
    )

In [ ]:
import os
import shutil
import cv2
import imageio


def turn_video_image_imageio(video_path: str, save_path: str, FRAME_RATE: int = 10):
    if not os.path.exists(video_path):
        print(f"Error: Video file not found at {video_path}")
        return

    print(f"Processing video with imageio: {os.path.basename(video_path)}")

    # Prepare save directory
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    os.makedirs(save_path, exist_ok=True)
    print(f"  Saving extracted images to: {save_path}")

    frames_saved = 0
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        meta = reader.get_meta_data()
        fps = meta['fps']
        total_frames = meta['nframes']
        duration = meta['duration']  # seconds

        print(f"  Original video FPS (imageio): {fps}")
        print(f"  Total frames in video (imageio): {total_frames}")
        print(f"  Video duration (imageio): {duration:.2f} seconds")
        print(f"  Desired extraction rate: {FRAME_RATE} frame(s) per second")

        # Save 1 frame every N frames based on original FPS and desired frame rate
        step = max(int(round(fps / FRAME_RATE)), 1)

        for i, frame in enumerate(reader):
            if i % step != 0:
                continue

            current_timestamp_sec = i / fps

            minutes = int(current_timestamp_sec // 60)
            remaining_seconds_float = current_timestamp_sec % 60
            seconds = int(remaining_seconds_float)
            milliseconds = int((remaining_seconds_float - seconds) * 1000)

            image_filename = f"{minutes:02d}_{seconds:02d}_{milliseconds:03d}.png"
            output_path = os.path.join(save_path, image_filename)

            try:
                cv2.imwrite(output_path, frame)
                frames_saved += 1
            except Exception as e:
                print(f"  Error saving frame at {current_timestamp_sec:.2f}s to {output_path}: {e}")

    except Exception as e:
        print(f"An error occurred with imageio during video processing: {e}")
        print("This might indicate an issue with FFmpeg installation or codec support for imageio.")

    print(f"  Successfully extracted {frames_saved} frames from {os.path.basename(video_path)}.")
    print("  Video processing complete for this file.")


In [ ]:
video_path_original = '/project/ai901504-ai0004/kaggle_competition/week7/train/Tom Cruise Heated Interview/Tom Cruise_s Heated Interview With Matt Lauer  Archives  TODAY.mp4'
save_path_imageio = '/project/ai901504-ai0004/501641_Big/week7/image/Tom Cruise_s Heated Interview With Matt Lauer Archives TODAY'

turn_video_image_imageio(video_path=video_path_original, save_path=save_path_imageio)

In [ ]:
def time_str_to_int(tstr):
    """Convert 'tMMSS' to total seconds"""
    minutes = int(tstr[1:3])
    seconds = int(tstr[3:])
    return minutes * 60 + seconds

train_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week7/train_pdf_concat.csv")

time_list = [time_str_to_int(i) for i in train_df['Time']]

train_df['subject_timestamp_pair'] = list(zip(train_df['Filename'], time_list))
unique_pairs = set(train_df['subject_timestamp_pair'].unique())


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def post_estimation_to_tabular(image_folder_path: str, unique_pairs: set):

    i = -1

    model_yolo = YOLO("/project/ai901504-ai0004/501641_Big/week7/yolo11x-pose.pt")
    df_columns = ['subject', 'minute', 'second', 'millisecond'] + [f'pos_{i}_{coord}' for i in range(17) for coord in ['x', 'y']]
    df = pd.DataFrame(columns=df_columns)
    subject_name = os.path.basename(image_folder_path.strip('/'))

    all_new_rows = []

    # Build expanded time window set
    buffer_seconds = 10
    expanded_pairs = set()

    for subj, ts in unique_pairs:
        for offset in range(-buffer_seconds, buffer_seconds + 1):
            expanded_pairs.add((subj, ts + offset))

    for image_file in os.listdir(image_folder_path):

        # Get only file name
        file_name_parts = os.path.splitext(image_file)[0].split('_')
        minute = int(file_name_parts[0])
        second = int(file_name_parts[1])
        millisecond = int(file_name_parts[2])
        timestamp_seconds = minute * 60 + second

        if (subject_name, timestamp_seconds) not in expanded_pairs:
            continue

        # YOLO Prediction
        image_path = os.path.join(image_folder_path, image_file)
        results = model_yolo(image_path, verbose=False, device='cuda')

        if (subject_name, timestamp_seconds) in unique_pairs:
            i += 1
            if (i % 150 == 0):
                # METHOD 1: Save and display using matplotlib (Jupyter-safe)
                annotated_frame = results[0].plot()
                # Convert for matplotlib display
                annotated_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
                
                # Display in Jupyter
                plt.figure(figsize=(10, 6))
                plt.imshow(annotated_rgb)
                plt.title(f"Pose Detection - {file_name_parts}")
                plt.axis('off')
                plt.tight_layout()
                plt.show()


        # วนลูปผ่านผลลัพธ์การตรวจจับ
        for result in results:

            if result.keypoints is None or result.keypoints.xy is None or len(result.keypoints.xy) == 0:
                continue

            # Get confidence score
            confidence_scores = result.boxes.conf

            # Get the highest confidence index
            if len(confidence_scores) == 0:
                continue

            max_conf_index = torch.argmax(confidence_scores)

            # highest_conf_pos_xy จะมี shape เป็น (17, 2) สำหรับ 17 Keypoints (x,y)
            highest_conf_pos_xy = result.keypoints.xy[max_conf_index]
            keypoints_conf = result.keypoints.conf[max_conf_index]  # (17,)

            # Build final keypoint list with NaN where confidence is low or missing
            flattened_keypoints = []
            for j in range(17):
                x, y = highest_conf_pos_xy[j].tolist()
                if keypoints_conf[j] > 0.3:  # You can change 0.3 as your confidence threshold
                    flattened_keypoints.extend(highest_conf_pos_xy[j].tolist())
                else:
                    if (x == 0.0 and y == 0.0):
                        flattened_keypoints.extend([float('nan'), float('nan')])
                    else:
                        flattened_keypoints.extend([x, y])

            # สร้างแถวใหม่สำหรับ DataFrame โดยเพิ่ม 'subject_name' เข้าไปในข้อมูล
            new_row_data = [subject_name, minute, second, millisecond] + flattened_keypoints
            all_new_rows.append(new_row_data)

    # หลังจากวนลูปทั้งหมดเสร็จสิ้น ค่อยสร้าง DataFrame จากลิสต์ของแถวทั้งหมด
    df = pd.DataFrame(all_new_rows, columns=df_columns)
    df = df.sort_values(by=['minute', 'second', 'millisecond']).reset_index(drop=True)
    df['Time'] = 't' + df['minute'].astype(str).str.zfill(2) + df['second'].astype(str).str.zfill(2)

    return df

In [ ]:
list_dataframe = []

for image_folder in glob('/project/ai901504-ai0004/501641_Big/week7/image/*'):
  df = post_estimation_to_tabular(image_folder, unique_pairs)
  list_dataframe.append(df)


# Concat all dataframe in list
df = pd.concat(list_dataframe)
df

In [ ]:
df.to_parquet("/project/ai901504-ai0004/501641_Big/week7/pose_estimate.parquet")

In [ ]:
df.isnull().sum()

#### Test df

In [ ]:
DATASET_PATH = "/project/ai901504-ai0004/kaggle_competition/week7/test/test"
SAVE_IMAGE_BASE_PATH = "/project/ai901504-ai0004/501641_Big/week7/image_test"
FRAME_RATE = 10

videos_to_process = [
    # คุณคมสันต์ ลี
    {
        'video_sub_path': 's06_smiley.mp4',
        'save_folder_name': 's06_smiley'
    },
    # Tom Cruise Interview
    {
        'video_sub_path': 'Tom_Cruise_on_Mission_Impossible_and_a_lifetime_of_learning_from_movies_BFI_in_Conversation.mp4',
        'save_folder_name': 'Tom_Cruise_on_Mission_Impossible_and_a_lifetime_of_learning_from_movies_BFI_in_Conversation'
    }
]


for video_info in tqdm(videos_to_process):
    video_full_path = os.path.join(DATASET_PATH, video_info['video_sub_path'])
    save_full_path = os.path.join(SAVE_IMAGE_BASE_PATH, video_info['save_folder_name'])
    
    turn_video_image_framerate(
        video_path=video_full_path,
        save_path=save_full_path,
        FRAME_RATE=FRAME_RATE
    )
    

In [ ]:
test_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week7/test_submission.csv")

time_list = [time_str_to_int(i) for i in test_df['Time']]

test_df['subject_timestamp_pair'] = list(zip(test_df['Filename'], time_list))
unique_pairs = set(test_df['subject_timestamp_pair'].unique())

In [ ]:
list_dataframe_test = []

for image_folder in glob('/project/ai901504-ai0004/501641_Big/week7/image_test/*'):
  df_test = post_estimation_to_tabular(image_folder, unique_pairs)
  list_dataframe_test.append(df_test)


# Concat all dataframe in list
df_test = pd.concat(list_dataframe_test)
df_test

In [ ]:
# Concat all dataframe in list
df_test = pd.concat(list_dataframe_test)
df_test

In [ ]:
df_test.to_parquet("/project/ai901504-ai0004/501641_Big/week7/pose_estimate_test.parquet")

In [ ]:
df.isnull().sum()

## Download for Google Sheet

In [ ]:
def download_google_sheet_to_csv(sheet_url, output_folder, sheet_name: dict):

  if not os.path.exists(output_folder):
    os.makedirs(output_folder)

  base_url_template = sheet_url.split('/edit')[0] + '/export?format=csv&gid='

  for sheet_name, gid in sheet_name.items():
    csv_export_url = f"{base_url_template}{gid}"
    output_file_path = os.path.join(output_folder, f"{sheet_name}.csv")
    df = pd.read_csv(csv_export_url)
    df.to_csv(output_file_path, index=False)

sheet_name = {
    'original': '2133183904',
    'Benedict Cumberbatch Interview': '1498105655',
    'Christianne Amanpour Interview' : '406456045',
    'Gary Oldman Interview' : '777905976',
    'Rose Interview(No Vid)' : '1020217620',
    'คุณคมสันต์ ลี' : '210771533',
    'Tom Cruise Heated Interview' : '103444326'
}

download_google_sheet_to_csv(
    sheet_url='https://docs.google.com/spreadsheets/d/1et6g9P3mnkKi8DXc9ZaoPJafogd5zgVS03wtwNb8n6c/edit',
    output_folder='/content/label',
    sheet_name=sheet_name
)

list_dataframe = []

for label_folder in glob('/content/label/*'):
  df_temp = pd.read_csv(label_folder)
  list_dataframe.append(df_temp)

del df_temp
del list_dataframe 

# Concat all dataframe in list
df_label = pd.concat(list_dataframe)
df_label = df_label.fillna(0)
df_label

# Experiment 1 : KAN Model

In [ ]:
!nvidia-smi

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
df_yolo_post = pd.read_parquet("/project/ai901504-ai0004/500101-Boss/week-7-action-recognition/dataset/df_yolo_post_17_fixed.parquet")
df_yolo_post

In [ ]:
df_yolo_post.info()

In [ ]:
def calculate_pos_speed(df):

    # For solve bug
    df = df.reset_index(drop=True)

    # Create timestanp
    if 'timestamp' not in df.columns or not pd.api.types.is_numeric_dtype(df['timestamp']):
        print("Creating 'timestamp' from 'minute', 'second', 'millisecond'.")
        df['timestamp'] = df['minute'] * 60 + df['second'] + df['millisecond'] / 1000

    # Get all position
    pos_base_cols = sorted(list(set([col.replace('_x', '').replace('_y', '') for col in df.columns if 'pos_' in col and ('_x' in col or '_y' in col)])))

    # คำนวณความเร็วสำหรับแต่ละตำแหน่ง
    for pos_base in pos_base_cols:
        x_col = f'{pos_base}_x'
        y_col = f'{pos_base}_y'
        speed_col_name = f'{pos_base}_speed'

        # ตรวจสอบว่าคอลัมน์ x และ y มีอยู่จริง
        if x_col not in df.columns or y_col not in df.columns:
            print(f"Skipping {pos_base}: Missing either {x_col} or {y_col} column.")
            continue

        # การคำนวณความเร็วภายในกลุ่ม subject ใช้ transform แทน apply เพื่อให้ได้ Series ที่มี Index ตรงกับ DataFrame หลัก
        df[speed_col_name] = df.groupby('subject').apply(lambda group: np.sqrt( (group[x_col].diff())**2 + (group[y_col].diff())**2 ) / group['timestamp'].diff())
        
        # reset_index เพื่อให้ Index กลับไปตรงกับ group
        df = df.reset_index(level=0, drop=True)

        # จัดการค่า NaN ในแถวแรกของแต่ละ subject (เนื่องจากไม่มีข้อมูลก่อนหน้า)
        df[speed_col_name] = df[speed_col_name].fillna(0)

    return df

In [ ]:
calculate_pos_speed(df_yolo_post)

In [ ]:
df_yolo_post.columns

In [ ]:
df_label = pd.read_csv("/project/ai901504-ai0004/500101-Boss/week-7-action-recognition/dataset/df_label_fixed.csv")
df_label = df_label.rename(columns={'Filename': 'subject'})
df_label

In [ ]:
df_label.columns

In [ ]:
target_columns = [
    'Advancing', 'Retreating', 'Enclosing', 'Spreading', 'Rising',
    'Descending', 'Directing', 'Indirecting', 'Increasing Pressure',
    'Decreasing Pressure', 'Acceleration', 'Decelerating', 'Accelerating'
]
train_columns = ['pos_0_x', 'pos_0_y',
     'pos_1_x', 'pos_1_y', 'pos_2_x', 'pos_2_y', 'pos_3_x', 'pos_3_y',
     'pos_4_x', 'pos_4_y', 'pos_5_x', 'pos_5_y', 'pos_6_x', 'pos_6_y',
       'pos_7_x', 'pos_7_y', 'pos_8_x', 'pos_8_y', 'pos_9_x', 'pos_9_y',
       'pos_10_x', 'pos_10_y', 'pos_11_x', 'pos_11_y', 'pos_12_x', 'pos_12_y',
       'pos_13_x', 'pos_13_y', 'pos_14_x', 'pos_14_y', 'pos_15_x', 'pos_15_y',
       'pos_16_x', 'pos_16_y'
]


# Merge Dataframe
df_train = pd.merge( 
    df_label, 
    df_yolo_post,
    on=['Time', 'subject'], 
    how='left'
)

# Fill 0
df_train = df_train.fillna(0)

# Create columns for target if non activity
df_train['is_activity'] = np.where((df_train[target_columns] != 0).any(axis=1), 1, 0)

# Drop Unused in train
df_train = df_train.drop(
    columns=[
        'Unnamed: 0', 'subject', 'minute', 'second', 'millisecond', 'Time', 'is_activity'
    ]
)

df_train

In [ ]:
df_train.columns

In [ ]:
# Configures Model
INPUT_FEATURS = len(train_columns)
NUM_FIRST_HIDDEN_LAYERS = 5
OUTPUT_NN = len(target_columns)
GRID_SIZE  = 3
DEGREE_OF_SPLINE = 3


# Calling Model
model_kan = KAN(
    width = [INPUT_FEATURS, NUM_FIRST_HIDDEN_LAYERS, OUTPUT_NN], 
    grid = GRID_SIZE , 
    k = DEGREE_OF_SPLINE, 
    seed = 42,
    device = device
)

model_kan

In [ ]:
df_train.info()

In [ ]:
# Turn to numpy
X_train_tensor = df_train[train_columns].values
X_train_tensor = torch.tensor(X_train_tensor, dtype=next(model_kan.parameters()).dtype)
X_train_tensor

In [ ]:
# plot KAN at initialization
model_kan(X_train_tensor)
model_kan.plot()